# Project Aeon: Quaternary Archival Storage, Reed-Solomon FEC, TMR & ML Reliability Predictor

**Author:** Md Shabbir Mia  
**Domain:** Coding Theory, Forward Error Correction (GF(2^8) Reed-Solomon), Triple Modular Redundancy (TMR), and Supervised Machine Learning for Channel Reliability Prediction.

This research-grade notebook provides a reproducible end-to-end workflow:
1. **Quaternary Symbol Transcoding** — Standard 2-bit (`00->A, 01->C, 10->G, 11->T`) vs. **Goldman Base-3 Rotational Coding** (zero consecutive identical symbols, `max_run = 1`).
2. **Canonical Huffman Compression** — Lossless source coding prior to channel protection.
3. **Cascaded FEC Architecture** — `RS(255, 105)` (`nsym = 150`, correcting up to 75 corrupted bytes/block) combined with 3-replica **Triple Modular Redundancy (TMR)**.
4. **Stochastic Channel-Noise Benchmarking** — Empirical recovery curves under uniform & transition-biased symbol substitutions.
5. **Supervised Machine Learning on CSV Dataset (`data/aeon_dna_stability_dataset.csv`)** — Exploratory Data Analysis, Feature Engineering, training a 3-Layer MLP & Random Forest classifier, evaluating **Test Accuracy, ROC-AUC, Precision, Recall, F1, Confusion Matrix**, and exporting pretrained JSON weights (`models/aeon_dna_stability_model.json`).
6. **Lab-Ready 80-Column `.fasta` Serialization & Image Recovery Proof-of-Concept**.

In [ ]:
import os, sys, json, math, csv
from pathlib import Path

# Ensure project root and python/ package are importable
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'python'))

import aeon_engine as aeon
import train_aeon_ml as ml_trainer

print('Project Root:', ROOT)
print('Aeon Engine loaded successfully.')

## Section 1: Quaternary Transcoding, Huffman Source Coding, RS(255, 105) & TMR Roundtrip

We encode an archival text payload using:
- **Canonical Huffman** lossless compression
- **Reed-Solomon GF(2^8)** with `nsym = 150` parity bytes per 255-byte codeword
- **Goldman Base-3 Rotational Mapping** (eliminating consecutive identical symbols)
- **Triple Modular Redundancy (3 replicas)**

We then inject **60 stochastic symbol substitutions** across the quaternary channel and verify **100% bit-perfect recovery**.

In [ ]:
sample_payload = (
    'Project Aeon Archival Benchmark: Silicon degrades in decades; '
    'quaternary error-corrected codewords preserve digital knowledge for millennia.'
)

dna_capsule, meta = aeon.create_text_capsule(
    sample_payload,
    rs_parity=150,
    scheme='goldman-base3',
    use_huffman=True,
)

# Verify zero consecutive identical symbols under Goldman Base-3
single_strand = dna_capsule[:meta.strand_length]
max_run = 1
cur = 1
for i in range(1, len(single_strand)):
    cur = cur + 1 if single_strand[i] == single_strand[i - 1] else 1
    max_run = max(max_run, cur)

print(f'Capsule ID       : {meta.capsule_id}')
print(f'Original Bytes   : {meta.original_length} B')
print(f'Compressed Bytes : {meta.encoded_length} B')
print(f'Strand Length    : {meta.strand_length} symbols (3x TMR = {len(dna_capsule)} total)')
print(f'Max Run Length   : {max_run} (Goldman guarantee == 1)')
print(f'Preview (60 nt)  : {single_strand[:60]}...')

# Apply 60 stochastic channel substitutions
corrupted = aeon.simulate_radiation(dna_capsule, mutations=60, seed=2026)
restored, stats = aeon.recover_text(corrupted, meta)

assert restored == sample_payload, 'Bit-perfect recovery assertion failed!'
print('\nRecovery Status  : 100% BIT-PERFECT MATCH')
print('Repair Telemetry :', stats)

## Section 2: Loading the Archival Channel Benchmark CSV Dataset

We load `data/aeon_dna_stability_dataset.csv` (1,600 trials spanning Standard 2-bit and Goldman Base-3 codewords, RS parity levels 16–150, TMR on/off, storage horizons 5–6,300 years, and temperatures -80°C to +35°C). If the CSV does not yet exist, we generate it deterministically.

In [ ]:
csv_path = ROOT / 'data' / 'aeon_dna_stability_dataset.csv'
if not csv_path.exists():
    ml_trainer.main()

try:
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(f'Loaded CSV shape: {df.shape}')
    display(df.head(8))
    print('\nClass distribution (degradation_risk_label):')
    print(df['degradation_risk_label'].value_counts(normalize=True).round(4))
    print('\nMean recovery accuracy by codec:')
    print(df.groupby('codec')[['max_homopolymer', 'degradation_risk_prob', 'recovered_accuracy_pct']].mean().round(3))
except ImportError:
    with open(csv_path, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'Loaded {len(rows)} rows using stdlib csv. Columns:', list(rows[0].keys()))
    print('Sample row:', rows[0])

## Section 3: Training & Evaluating the Supervised ML Models (MLP + Random Forest)

We extract 8 quantitative codeword & environmental features:
1. `gc_deviation` — $|\text{GC} - 0.50| \times 2$
2. `norm_homopolymer` — Normalized max consecutive identical symbol run
3. `purine_density` — Fraction of `{A, G}` symbols
4. `cpg_deamination_score` — `C` & `CG` motif density
5. `stacking_delta_g` — Normalized nearest-neighbor stability parameter
6. `storage_years` — $\log_{10}(\text{years} + 1) / 4$
7. `temperature_c` — Normalized storage temperature
8. `radiation_msv_yr` — Normalized channel noise intensity

We perform an **80/20 Train/Test split**, train the `AeonMLP (8 -> 16 -> 8 -> 1)` neural network, and benchmark it alongside `scikit-learn` (`RandomForestClassifier` & `LogisticRegression`) when installed.

In [ ]:
X, y = [], []
with open(csv_path, 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        feats, _, _, _ = ml_trainer.extract_features(
            row['sequence_window'],
            float(row['storage_years']),
            float(row['temperature_c']),
            float(row['radiation_msv_yr']),
        )
        X.append(feats)
        y.append(int(row['degradation_risk_label']))

split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

mlp = ml_trainer.AeonMLP(seed=42)
history = mlp.fit(X_train, y_train, epochs=60, lr=0.03, batch_size=32)
metrics = ml_trainer.evaluate(mlp, X_test, y_test)

print('=== Aeon-MLP (8 -> 16 -> 8 -> 1) Held-Out Test Evaluation ===')
print(f"Test Accuracy : {metrics['accuracy'] * 100:.2f}%")
print(f"ROC-AUC       : {metrics['rocAuc']:.4f}")
print(f"Precision     : {metrics['precision']:.4f}")
print(f"Recall        : {metrics['recall']:.4f}")
print(f"F1-Score      : {metrics['f1Score']:.4f}")
print(f"Brier Score   : {metrics['brierScore']:.4f}")
print(f"Confusion Mtx : {metrics['confusionMatrix']}")

try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
    rf = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)
    rf_probs = rf.predict_proba(X_test)[:, 1]
    print('\n=== Scikit-Learn RandomForest Benchmark ===')
    print(f'RF Test Accuracy : {accuracy_score(y_test, rf_preds) * 100:.2f}%')
    print(f'RF ROC-AUC       : {roc_auc_score(y_test, rf_probs):.4f}')
    print(classification_report(y_test, rf_preds, target_names=['Stable (0)', 'High-Risk (1)']))
except ImportError:
    print('\n(Optional) Install scikit-learn to also run the RandomForest baseline comparison.')

## Section 4: Predicting Reliability & Exporting Pretrained Model Artifact + FASTA

We inspect the saved pretrained JSON model (`models/aeon_dna_stability_model.json`), run live inference comparing a **Goldman Base-3 Cryogenic Archive** against an **Unbalanced Homopolymer Warm Archive**, and export the NCBI-compatible 80-column `.fasta` sequence.

In [ ]:
good_feats, _, _, _ = ml_trainer.extract_features(
    'ACGTACGTGCATGCATACGTACGTGCATGCAT', storage_years=1000, temp_c=-20, radiation_msv=1.5
)
bad_feats, _, _, _ = ml_trainer.extract_features(
    'AAAAAAAAAAAAAAAATTTTTTTTTTTTTTTT', storage_years=2500, temp_c=25, radiation_msv=18.0
)

print(f'Goldman Cryo Risk Probability     : {mlp.predict(good_feats) * 100:.2f}% (Recommended)')
print(f'Homopolymer Warm Risk Probability : {mlp.predict(bad_feats) * 100:.2f}% (High Risk)')

fasta_text = aeon.export_fasta(dna_capsule, meta)
print('\n=== 80-Column FASTA Preview ===')
print('\n'.join(fasta_text.splitlines()[:8]))